# Deep Generative Prior (DGP) for the inverse problem

Uses the WGAN-GP generator trained on the clean images (notebook 1) as a prior to
reconstruct a clean image from a blurred + noisy measurement `y`.

The EMA generator weights (`generator_ema`) are used when present in the
checkpoint (they give smoother reconstructions), otherwise the standard
`generator` weights.

For each measurement `y_delta` the reconstruction runs in two stages:

1. **Latent optimization** — with the generator frozen, optimize only the latent
   `z` so that the blurred generation matches the measurement:
   `min_z ||K(G(z)) - y_delta||^2 + lambda_z ||z||^2`

2. **Generator fine-tuning (the DGP step)** — then relax the generator: jointly
   optimize `(z, theta_G)` on the same data term plus a penalty that keeps the
   weights close to the trained prior. This lets `G` adapt to the specific image
   (which pure latent search cannot reach) without turning into an unconstrained
   decoder, giving a markedly sharper reconstruction.

`K` is the same blur operator used in `0.5 Data_Degradation.ipynb`. The noisy
columns `y_005`, `y_010`, `y_050`, `y_100` are used only as measurements (never as
training targets); the ground truth `x` is used only to report PSNR/SSIM.


## Imports and paths

In [ ]:
from __future__ import annotations

# Metrics + standard library
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import json
import math
import os
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

# Scientific / deep-learning stack
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from datasets import DatasetDict, load_from_disk
from tqdm.auto import tqdm


# Paths. On Google Colab, mount Drive and use the project folder there; running
# locally instead, fall back to the repository root. Either way the structure is
# the same: BASE_DIR / "gan_output" / {weights, samples, metrics_data, reconstruction_examples}.
try:
    from google.colab import drive
    !pip install astra-toolbox
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Computational Imaging")
except ModuleNotFoundError:
    _cwd = Path.cwd().resolve()
    BASE_DIR = next((c for c in (_cwd, _cwd.parent) if (c / "Homeworks").exists()), Path("..").resolve())
DATASET_DIR = BASE_DIR / "dataset_64x64"

# Outputs share the gan_output/ tree created by the training notebook
OUTPUT_DIR  = BASE_DIR / "gan_output"
WEIGHTS_DIR = OUTPUT_DIR / "weights"                       # written by file 1
CHECKPOINT_PATH = WEIGHTS_DIR / "checkpoint_last.pt"       # trained generator to load
METRICS_DIR = OUTPUT_DIR / "metrics_data"                  # data-loss curves
RECON_DIR = OUTPUT_DIR / "reconstruction_examples"         # reconstruction figures
for _d in (METRICS_DIR, RECON_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# make IPPy importable, then bring in the blur forward operator
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from IPPy.IPPy.operators import Blurring


def get_device() -> torch.device:
    """ Use the GPU if available, otherwise CPU """
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
print("Project root:", BASE_DIR)
print("Dataset:", DATASET_DIR)
print("Checkpoint:", CHECKPOINT_PATH)
print("Device:", DEVICE)


## Configuration

In [ ]:
# Noise levels present in the dataset: key -> sigma of the additive noise
NOISE_LEVEL_KEYS: tuple[str, ...] = ("005", "010", "050", "100")
NOISE_LEVEL_TO_SIGMA: dict[str, float] = {
    "005": 0.005,
    "010": 0.01,
    "050": 0.05,
    "100": 0.1,
}


@dataclass(frozen=True)
class ForwardConfig:
    """ Parameters of the forward (degradation) operator K: a Gaussian blur. """
    img_shape: tuple[int, int, int] = (3, 64, 64)
    blur_kernel_type: str = "gaussian"
    blur_kernel_size: int = 9
    blur_sigma: float = 2.0


@dataclass(frozen=True)
class LatentOptConfig:
    """ Settings for the reconstruction of one sample: stage-1 latent optimization
    and stage-2 DGP generator fine-tuning. """
    split: str = "test"            # dataset split to read the image from
    sample_index: int = 50         # which image in the split
    noise_key: str = "100"         # which degraded version y_* to reconstruct from
    steps: int = 800               # stage-1 latent optimization steps
    lr: float = 0.02               # stage-1 Adam LR on z
    restarts: int = 2              # stage-1 random restarts (keep best)
    z_l2_weight: float = 1e-4      # weak prior pulling z towards the origin
    # --- Stage 2: DGP generator fine-tuning (relax G to fit THIS image) ---
    dgp_finetune: bool = True      # enable the generator-relaxation stage
    dgp_steps: int = 600           # stage-2 joint (z, theta_G) steps
    dgp_lr_z: float = 1e-3         # stage-2 LR on z
    dgp_lr_g: float = 2e-4         # stage-2 LR on the generator weights
    dgp_weight_reg: float = 0.05   # stay-close-to-prior strength
    report_every: int = 25         # logging / metric interval
    seed: int = 123
    save_figures: bool = True

    def __post_init__(self):
        # guard against an invalid noise level
        if self.noise_key not in NOISE_LEVEL_KEYS:
            raise ValueError(f"noise_key must be one of {NOISE_LEVEL_KEYS}, got {self.noise_key!r}")


def set_seed(seed: int = 42) -> None:
    """ Set every RNG for reproducibility """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


## Dataset utilities

In [ ]:
pil_to_tensor = T.ToTensor()   # PIL image -> [0, 1] tensor


def load_id2label(dataset_dir: Path = DATASET_DIR) -> dict[int, str]:
    """ Read labels.json (raw class id -> human readable name); {} if missing """
    labels_path = dataset_dir / "labels.json"
    if not labels_path.exists():
        return {}
    with open(labels_path) as f:
        return {int(k): v for k, v in json.load(f).items()}


def load_degraded_dataset(dataset_dir: Path = DATASET_DIR) -> DatasetDict:
    """ Load the dataset and verify it has the clean image, label and all y_* columns """
    ds_dict: DatasetDict = load_from_disk(str(dataset_dir))
    required_columns = {"x", "label", *(f"y_{key}" for key in NOISE_LEVEL_KEYS)}
    missing = required_columns - set(ds_dict["train"].column_names)
    if missing:
        raise ValueError(f"Dataset is missing columns: {sorted(missing)}")
    return ds_dict


def load_sample(
    ds_dict: DatasetDict,
    split: str,
    sample_index: int,
    noise_key: str,
    device: torch.device = DEVICE,
) -> dict[str, Any]:
    """
    Load one image as a batch of size 1: the clean target and its degraded measurement.

    Parameters:
      ds_dict: the loaded DatasetDict
      split: "train" / "validation" / "test"
      sample_index: row index within the split
      noise_key: which degraded version to use as the measurement y
      device: device to place the tensors on
    """
    row = ds_dict[split][sample_index]
    x_true = pil_to_tensor(row["x"]).unsqueeze(0).to(device)            # ground truth
    y_delta = pil_to_tensor(row[f"y_{noise_key}"]).unsqueeze(0).to(device)  # measurement
    return {
        "x_true": x_true,
        "y_delta": y_delta,
        "label": int(row["label"]),
        "split": split,
        "sample_index": sample_index,
        "noise_key": noise_key,
    }


def tensor_to_image(image: torch.Tensor) -> np.ndarray:
    """ (1,C,H,W) torch tensor -> (H,W,C) numpy array in [0, 1] for plotting/metrics """
    image = image.detach().cpu().squeeze(0).permute(1, 2, 0).clamp(0, 1)
    return image.numpy()


def mse_value(a: torch.Tensor, b: torch.Tensor) -> float:
    """ Plain MSE between two tensors, as a python float """
    return float(F.mse_loss(a.detach(), b.detach()).cpu())


def compute_metrics(x_true: torch.Tensor, x_hat: torch.Tensor) -> tuple[float, float]:
    """Calcola PSNR e SSIM usando skimage sulle immagini convertite in numpy."""
    img_true = tensor_to_image(x_true)
    img_hat = tensor_to_image(x_hat)

    # skimage si aspetta array (H, W, C). channel_axis=-1 gestisce correttamente i canali RGB.
    psnr_val = peak_signal_noise_ratio(img_true, img_hat, data_range=1.0)
    ssim_val = structural_similarity(img_true, img_hat, data_range=1.0, channel_axis=-1)

    return float(psnr_val), float(ssim_val)


In [ ]:
# Load the degraded dataset and the class names, then print a quick summary
ds_dict = load_degraded_dataset()
id2label = load_id2label()

print("Splits:", {split: len(ds_dict[split]) for split in ds_dict.keys()})
print("Columns:", ds_dict["train"].column_names)
print("Classes:", id2label)


## Recreate and load the trained GAN generator

In [ ]:
class ResBlockUp(nn.Module):
    """Upsample 2x with residual connection (nearest-neighbor interpolation).
    The generator architecture must match the one the weights were trained with."""
    def __init__(self, in_channels: int, out_channels: int):
        """
        Parameters:
          in_channels:  channels of the input feature map
          out_channels: channels produced by the block
        """
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # skip path: upsample then 1x1 conv
        residual = F.interpolate(x, scale_factor=2, mode="nearest")
        residual = self.skip(residual)
        # main path: BN-ReLU, upsample, two 3x3 convs
        out = F.relu(self.bn1(x), inplace=False)
        out = F.interpolate(out, scale_factor=2, mode="nearest")
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out), inplace=False))
        return out + residual


class GoodGenerator(nn.Module):
    """Latent generator G(z) -> clean RGB image in [-1, 1]. Must be identical to
    the architecture the checkpoint was trained with, so the weights load."""

    def __init__(self, latent_dim: int = 128, base_channels: int = 64, image_channels: int = 3):
        """
        Parameters:
          latent_dim:     dimension of the input noise z
          base_channels:  width multiplier
          image_channels: output channels (3 for RGB)
        """
        super().__init__()
        self.latent_dim = latent_dim
        self.base_channels = base_channels
        # project z to 4x4, then upsample four times to 64x64
        self.fc = nn.Linear(latent_dim, 4 * 4 * 8 * base_channels)
        self.blocks = nn.Sequential(
            ResBlockUp(8 * base_channels, 8 * base_channels),
            ResBlockUp(8 * base_channels, 4 * base_channels),
            ResBlockUp(4 * base_channels, 2 * base_channels),
            ResBlockUp(2 * base_channels, base_channels),
        )
        self.bn = nn.BatchNorm2d(base_channels)
        self.conv_out = nn.Conv2d(base_channels, image_channels, kernel_size=3, padding=1)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        out = self.fc(z).view(z.size(0), 8 * self.base_channels, 4, 4)
        out = self.blocks(out)
        out = self.conv_out(F.relu(self.bn(out), inplace=False))
        return torch.tanh(out)   # bound output to [-1, 1]


def load_checkpoint(path: Path, map_location: torch.device | str = "cpu") -> dict[str, Any]:
    """ Load a checkpoint dict, tolerating older torch without weights_only """
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_generator_from_checkpoint(
    checkpoint_path: Path = CHECKPOINT_PATH,
    device: torch.device = DEVICE,
) -> tuple[GoodGenerator, dict[str, Any]]:
    """
    Rebuild the generator from the architecture saved in the checkpoint config and
    load its (EMA, if present) weights. Returned generator is frozen and in eval mode.

    Parameters:
      checkpoint_path: file written by the training notebook
      device: device to load onto
    """
    checkpoint = load_checkpoint(checkpoint_path, map_location=device)
    # read the architecture sizes back from the saved config
    gan_config = checkpoint.get("config", {})
    latent_dim = int(gan_config.get("latent_dim", 128))
    base_channels = int(gan_config.get("base_channels", 64))
    image_channels = int(gan_config.get("image_channels", 3))

    generator = GoodGenerator(
        latent_dim=latent_dim,
        base_channels=base_channels,
        image_channels=image_channels,
    ).to(device)

    # Prefer EMA weights if available, fall back to standard generator weights
    if "generator_ema" in checkpoint:
        generator.load_state_dict(checkpoint["generator_ema"])
        print("Loaded generator weights: generator_ema (EMA)")
    else:
        generator.load_state_dict(checkpoint["generator"])
        print("Loaded generator weights: generator (standard — no EMA found)")

    # freeze: stage 1 only optimizes z (stage 2 will clone & unfreeze a copy)
    generator.eval()
    for param in generator.parameters():
        param.requires_grad_(False)
    return generator, {"checkpoint": checkpoint, "gan_config": gan_config, "latent_dim": latent_dim}


In [ ]:
# Load the trained generator and read back its latent size
generator, generator_info = load_generator_from_checkpoint()
latent_dim = generator_info["latent_dim"]
checkpoint = generator_info["checkpoint"]
print("Loaded epoch:", checkpoint.get("epoch"))
print("GAN config:", generator_info["gan_config"])
print("Latent dim:", latent_dim)


## Forward operator K

In [ ]:
def build_forward_operator(config: ForwardConfig = ForwardConfig()) -> Blurring:
    """ Build the Gaussian-blur forward operator K (same as used in notebook 0.5) """
    return Blurring(
        img_shape=config.img_shape,
        kernel_type=config.blur_kernel_type,
        kernel_size=config.blur_kernel_size,
        kernel_variance=config.blur_sigma ** 2,
    )


# instantiate K once and reuse it for every reconstruction
forward_config = ForwardConfig()
K = build_forward_operator(forward_config)
print(forward_config)


## Latent optimization + DGP generator fine-tuning


In [ ]:
import copy


def generator_to_01(generator_output: torch.Tensor) -> torch.Tensor:
    """ Map generator output from [-1, 1] to [0, 1] (the range K and y live in) """
    return (generator_output + 1.0) / 2.0


def sample_latent(batch_size: int, latent_dim: int, device: torch.device = DEVICE) -> torch.Tensor:
    """ Draw standard-normal latent vectors z ~ N(0, I) """
    return torch.randn(batch_size, latent_dim, device=device)


def _psnr_ssim(x_hat, x_true):
    """ PSNR/SSIM of x_hat vs x_true (NaN if no ground truth is given) """
    if x_true is None:
        return math.nan, math.nan
    return compute_metrics(x_true, x_hat)


def latent_optimization(generator, K, y_delta, x_true, latent_dim, config, device):
    """Stage 1 - optimize z only (generator frozen). Returns (best_z, all_histories).

    Searches the latent space for the z whose blurred generation K(G(z)) best
    matches the measurement y. Runs several restarts and keeps the best z.

    Parameters:
      generator: frozen trained generator
      K: forward (blur) operator
      y_delta: the measurement (degraded image)
      x_true: ground truth, used only for logging metrics (not in the loss)
      latent_dim: size of z
      config: LatentOptConfig (uses steps, lr, restarts, z_l2_weight, report_every)
      device: compute device
    """
    best_z, best_data, all_hist = None, math.inf, []
    for restart in range(config.restarts):
        # fresh random z per restart, optimized with Adam + cosine LR decay
        z = sample_latent(1, latent_dim, device).requires_grad_(True)
        opt = torch.optim.Adam([z], lr=config.lr)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=config.steps, eta_min=config.lr * 0.05)
        hist = []
        prog = tqdm(range(config.steps), desc=f"z-opt {restart + 1}/{config.restarts}", leave=False)
        for step in prog:
            opt.zero_grad(set_to_none=True)
            x_hat = generator_to_01(generator(z))
            # data term: match the blurred generation to the measurement
            data_loss = F.mse_loss(K(x_hat), y_delta)
            loss = data_loss + config.z_l2_weight * z.pow(2).mean()
            loss.backward(); opt.step(); sch.step()
            # periodically record loss + metrics for the curves
            if step == 0 or (step + 1) % config.report_every == 0 or step == config.steps - 1:
                with torch.no_grad():
                    p, smetric = _psnr_ssim(x_hat, x_true)
                hist.append({"step": float(step + 1), "data_loss": float(data_loss.detach().cpu()),
                             "x_psnr": p, "x_ssim": smetric})
                prog.set_postfix(data=f"{data_loss.item():.3e}", psnr=f"{p:.2f}")
        # keep the restart with the lowest final data loss
        with torch.no_grad():
            fd = mse_value(K(generator_to_01(generator(z))), y_delta)
        all_hist.append(hist)
        if fd < best_data:
            best_data, best_z = fd, z.detach().clone()
    return best_z, all_hist


def dgp_finetune(generator, K, z_init, y_delta, x_true, config, device):
    """Stage 2 - Deep Generative Prior: relax the generator. Jointly optimize
    (z, theta_G) on the data term with a stay-close-to-prior penalty so G adapts
    to THIS image without collapsing into an unconstrained decoder. A clone of G
    is used so each image is independent; eval() keeps BN running stats (batch=1).

    Parameters:
      generator: the frozen trained generator (cloned internally, left untouched)
      K: forward (blur) operator
      z_init: latent from stage 1 to start from
      y_delta: the measurement
      x_true: ground truth, for logging only (not in the loss)
      config: LatentOptConfig (uses dgp_steps, dgp_lr_z, dgp_lr_g, dgp_weight_reg, ...)
      device: compute device

    Returns:
      (fine-tuned generator clone, final z, history)
    """
    # clone so the original prior is preserved; eval() -> BN uses running stats
    g = copy.deepcopy(generator)
    g.eval()
    for p in g.parameters():
        p.requires_grad_(True)
    # snapshot of the prior weights, for the stay-close penalty
    theta0 = [p.detach().clone() for p in g.parameters()]
    n_params = len(theta0)

    # optimize z and the generator weights together, with separate LRs
    z = z_init.clone().requires_grad_(True)
    opt = torch.optim.Adam([
        {"params": [z], "lr": config.dgp_lr_z},
        {"params": list(g.parameters()), "lr": config.dgp_lr_g},
    ])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=config.dgp_steps, eta_min=config.dgp_lr_g * 0.1)
    hist = []
    prog = tqdm(range(config.dgp_steps), desc="dgp-finetune", leave=False)
    for step in prog:
        opt.zero_grad(set_to_none=True)
        x_hat = generator_to_01(g(z))
        data_loss = F.mse_loss(K(x_hat), y_delta)
        # mean squared deviation of the weights from the prior (regularizer)
        reg = sum(((p - p0) ** 2).mean() for p, p0 in zip(g.parameters(), theta0)) / n_params
        loss = data_loss + config.z_l2_weight * z.pow(2).mean() + config.dgp_weight_reg * reg
        loss.backward(); opt.step(); sch.step()
        if step == 0 or (step + 1) % config.report_every == 0 or step == config.dgp_steps - 1:
            with torch.no_grad():
                p, sm = _psnr_ssim(x_hat, x_true)
            hist.append({"step": float(step + 1), "data_loss": float(data_loss.detach().cpu()),
                         "reg": float(reg.detach().cpu()), "x_psnr": p, "x_ssim": sm})
            prog.set_postfix(data=f"{data_loss.item():.3e}", reg=f"{reg.item():.2e}", psnr=f"{p:.2f}")
    return g, z.detach(), hist


def optimize_latent_for_sample(
    generator: GoodGenerator,
    K: Blurring,
    sample: dict[str, Any],
    latent_dim: int,
    config: LatentOptConfig,
    device: torch.device = DEVICE,
) -> dict[str, Any]:
    """
    Full DGP reconstruction of one sample: stage-1 latent optimization, then
    (optionally) stage-2 generator fine-tuning. Returns a result dict with the
    reconstruction, metrics and histories.

    Parameters:
      generator: frozen trained generator
      K: forward (blur) operator
      sample: dict from load_sample (x_true, y_delta, label, ...)
      latent_dim: size of z
      config: LatentOptConfig
      device: compute device
    """
    y_delta = sample["y_delta"].to(device)
    x_true = sample.get("x_true")
    if x_true is not None:
        x_true = x_true.to(device)

    set_seed(config.seed)
    t0 = time.time()

    # Stage 1: latent optimization (z only, G frozen)
    best_z, all_histories = latent_optimization(generator, K, y_delta, x_true, latent_dim, config, device)
    # reconstruction and metrics using z-only (for the before/after comparison)
    with torch.no_grad():
        x_hat_z = generator_to_01(generator(best_z)).clamp(0, 1)
    psnr_z, ssim_z = _psnr_ssim(x_hat_z, x_true)

    # Stage 2: DGP generator fine-tuning (z + theta_G)
    dgp_history = []
    final_gen = generator
    if config.dgp_finetune:
        final_gen, best_z, dgp_history = dgp_finetune(
            generator, K, best_z, y_delta, x_true, config, device)

    # final reconstruction (from the fine-tuned G if stage 2 ran) and metrics
    with torch.no_grad():
        x_hat = generator_to_01(final_gen(best_z)).clamp(0, 1)
        y_hat = K(x_hat).clamp(0, 1)
        final_data = mse_value(y_hat, y_delta)
        final_x_mse = mse_value(x_hat, x_true) if x_true is not None else math.nan
        final_x_psnr, final_x_ssim = _psnr_ssim(x_hat, x_true)

    # pack everything the plotting/printing helpers need
    return {
        "restart": 0,
        "z": best_z.detach().cpu(),
        "x_hat": x_hat.detach().cpu(),
        "y_hat": y_hat.detach().cpu(),
        "x_hat_z_only": x_hat_z.detach().cpu(),
        "final_data_loss": final_data,
        "final_x_mse": final_x_mse,
        "final_x_psnr": final_x_psnr,
        "final_x_ssim": final_x_ssim,
        "psnr_z_only": psnr_z,
        "ssim_z_only": ssim_z,
        "history": dgp_history if dgp_history else (all_histories[0] if all_histories else []),
        "all_histories": all_histories,
        "dgp_history": dgp_history,
        "seconds": time.time() - t0,
        "sample": {"split": sample["split"], "sample_index": sample["sample_index"],
                   "noise_key": sample["noise_key"], "label": sample["label"]},
    }


## Visualization

In [ ]:
def plot_history(result: dict[str, Any]) -> None:
    """ Plot the stage-1 data loss ||K(G(z)) - y||^2 over the restarts (log scale) """
    plt.figure(figsize=(7, 4))
    for restart, history in enumerate(result["all_histories"]):
        if not history:
            continue
        steps = [h["step"] for h in history]
        data_losses = [h["data_loss"] for h in history]
        plt.plot(steps, data_losses, label=f"restart {restart + 1}")
    plt.yscale("log")
    plt.xlabel("step")
    plt.ylabel("data loss ||K(G(z)) - y||^2")
    plt.legend()
    plt.tight_layout()
    _s = result["sample"]
    METRICS_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(METRICS_DIR / f"dgp_dataloss_{_s['split']}_{_s['sample_index']:04d}_y{_s['noise_key']}.png",
                dpi=140, bbox_inches="tight")
    plt.show()


def plot_reconstruction(
    sample: dict[str, Any],
    result: dict[str, Any],
    id2label: dict[int, str] | None = None,
    save_dir: Path | None = None,
) -> Path | None:
    """
    Show the 5-panel reconstruction figure and optionally save it.

    Panels: x true | noisy measurement y | G(z*) | K(G(z*)) | residual |K(G(z*))-y|.

    Parameters:
      sample: dict from load_sample
      result: dict from optimize_latent_for_sample
      id2label: optional id -> class name map for the title
      save_dir: if given, the figure is written there as a PNG
    """
    # gather the five images to display
    x_true = sample["x_true"].detach().cpu()
    y_delta = sample["y_delta"].detach().cpu()
    x_hat = result["x_hat"]
    y_hat = result["y_hat"]
    residual = (y_hat - y_delta).abs().clamp(0, 1)

    # resolve a readable class name and the noise sigma for the title
    label_id = sample["label"]
    label_name = id2label.get(label_id, str(label_id)) if id2label else str(label_id)
    noise_key = sample["noise_key"]
    sigma = NOISE_LEVEL_TO_SIGMA[noise_key]

    titles = [
        "x true",
        f"y noisy sigma={sigma}",
        "G(z*)",
        "K(G(z*))",
        "|K(G(z*)) - y|",
    ]
    images = [x_true, y_delta, x_hat, y_hat, residual]

    # draw the panels
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.4))
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(tensor_to_image(image))
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    fig.suptitle(
        f"{sample['split']}[{sample['sample_index']}] | {label_name} | "
        f"data MSE={result['final_data_loss']:.4e} | x PSNR={result['final_x_psnr']:.2f} dB | x SSIM={result['final_x_ssim']:.4f}",
        fontsize=10,
    )
    plt.tight_layout()

    # optionally save to disk
    saved_path = None
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        saved_path = save_dir / (
            f"dgp_{sample['split']}_{sample['sample_index']:04d}_y{noise_key}.png"
        )
        fig.savefig(saved_path, dpi=140, bbox_inches="tight")
        print("Saved figure:", saved_path)
    plt.show()
    return saved_path


def print_result_summary(result: dict[str, Any]) -> None:
    """ Print the reconstruction metrics, including z-only vs after DGP fine-tuning """
    sample = result["sample"]
    print("Sample:", sample)
    # before/after comparison when stage-2 metrics are available
    if not math.isnan(result.get("psnr_z_only", float("nan"))):
        print(f"PSNR z-only:        {result['psnr_z_only']:.2f} dB | SSIM {result['ssim_z_only']:.4f}")
        print(f"PSNR + DGP finetune: {result['final_x_psnr']:.2f} dB | SSIM {result['final_x_ssim']:.4f}")
    print(f"Data MSE: {result['final_data_loss']:.6e}")
    print(f"x MSE:    {result['final_x_mse']:.6e}")
    print(f"x PSNR:   {result['final_x_psnr']:.2f} dB")
    print(f"x SSIM:   {result['final_x_ssim']:.4f}")
    print(f"Seconds:  {result['seconds']:.1f}")


## Run one reconstruction

Start with one image and one noise level. The defaults are intentionally moderate. If the result is unstable, try more restarts before increasing the number of steps.


In [ ]:
# Reconstruct a handful of test images at one noise level
sample_indices = [100, 101, 102, 103, 104]
noise_key = "005"

for idx in sample_indices:
    print(f"\n" + "="*50)
    print(f"Processing sample_index: {idx}")
    print("="*50)

    # config for the current image
    opt_config = LatentOptConfig(noise_key=noise_key, sample_index=idx)
    sample = load_sample(ds_dict, opt_config.split, opt_config.sample_index, opt_config.noise_key)
    set_seed(opt_config.seed)

    # run stage-1 latent optimization + stage-2 DGP fine-tuning
    result = optimize_latent_for_sample(generator, K, sample, latent_dim, opt_config)

    # print metrics
    print_result_summary(result)

    # save / show the reconstruction and the loss curves
    save_dir = RECON_DIR if opt_config.save_figures else None
    plot_reconstruction(sample, result, id2label=id2label, save_dir=save_dir)
    plot_history(result)


## Input vs output grid (all noise levels)

For one test image, reconstruct it at every noise level and show a 2x4 grid:
the degraded inputs on top, the DGP reconstructions below.


In [ ]:
# 2x4 grid for a single image: degraded inputs (top) vs DGP reconstructions (bottom).
# One column per noise level, with the ground truth shown separately below.

GRID_SPLIT        = "test"   # split to take the image from
GRID_SAMPLE_INDEX = 158      # which image (change freely)

# ground truth (clean) image and its class name, for reference
gt_row = ds_dict[GRID_SPLIT][GRID_SAMPLE_INDEX]
img_clean = tensor_to_image(pil_to_tensor(gt_row["x"]).unsqueeze(0))
label_name = id2label.get(int(gt_row["label"]), str(gt_row["label"])) if id2label else str(gt_row["label"])

# reconstruct the image at each noise level and fill the grid
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for col, nk in enumerate(NOISE_LEVEL_KEYS):
    # full DGP reconstruction (stage-1 latent opt + stage-2 fine-tuning)
    cfg = LatentOptConfig(split=GRID_SPLIT, sample_index=GRID_SAMPLE_INDEX, noise_key=nk)
    sample = load_sample(ds_dict, GRID_SPLIT, GRID_SAMPLE_INDEX, nk)
    set_seed(cfg.seed)
    result = optimize_latent_for_sample(generator, K, sample, latent_dim, cfg)

    img_input  = tensor_to_image(sample["y_delta"])   # degraded measurement
    img_output = tensor_to_image(result["x_hat"])     # reconstruction G(z*)

    # top row: degraded input at this noise level
    axes[0, col].imshow(img_input)
    axes[0, col].set_title(f"Input y_{nk} (sigma={NOISE_LEVEL_TO_SIGMA[nk]})",
                           fontsize=14, fontweight="bold")
    axes[0, col].axis("off")

    # bottom row: DGP output, annotated with its PSNR
    axes[1, col].imshow(img_output)
    axes[1, col].set_title(f"Output y_{nk}  -  PSNR {result['final_x_psnr']:.2f} dB",
                           fontsize=14, fontweight="bold", color="green")
    axes[1, col].axis("off")

fig.suptitle(f"{GRID_SPLIT}[{GRID_SAMPLE_INDEX}]  -  {label_name}", fontsize=16, fontweight="bold")
plt.tight_layout()

# save the grid next to the other reconstructions
RECON_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(RECON_DIR / f"dgp_grid_{GRID_SPLIT}_{GRID_SAMPLE_INDEX:04d}.png", dpi=200, bbox_inches="tight")
plt.show()

# ground truth shown on its own (the inputs/outputs above are 64x64)
plt.figure(figsize=(5, 5))
plt.imshow(img_clean)
plt.title(f"Ground Truth  -  {label_name}", fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()
